[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/01_collection/A5_buildingeye_import.ipynb)

# A5: BuildingEye Data Import

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Import CSV data** exported from BuildingEye planning portal
2. **Parse permit dates** from various formats
3. **Match permits to projects** using address and permit number
4. **Populate the permit_events table** for timeline tracking
5. **Update project milestone dates** (first_filed, co_issued, etc.)

## Why This Matters

BuildingEye (berkeley.buildingeye.com/planning) provides the **date fields** we need to track
project timelines - from initial filing to Certificate of Occupancy. This data enables:
- Measuring how long projects take at each stage
- Identifying bottlenecks in the approval process
- Tracking which projects are stalled
- Calculating completion rates

## Data Source

**Manual CSV Export from BuildingEye:**
1. Go to https://berkeley.buildingeye.com/planning
2. Set date range (e.g., 2020-01-01 to present)
3. Select permit types: Zoning, Building, etc.
4. Click "Download CSV"
5. Save to `data/raw/buildingeye_export_YYYYMMDD.csv`

---

In [ ]:
# ============================================================================
# COLAB ENVIRONMENT SETUP (Run this first in Colab!)
# ============================================================================

import os
import sys
from pathlib import Path

print('SETTING UP ENVIRONMENT')
print('='*70)

# Detect environment
try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

if IN_COLAB:
    # Clone repository
    repo_path = Path('/content/berkeley-housing-analysis')
    
    if not repo_path.exists():
        print('\nCloning repository...')
        !git clone https://github.com/blockXblock/berkeley-housing-analysis.git
        print('Repository cloned')
    else:
        print('\nRepository already exists')
        # Pull latest changes
        !cd /content/berkeley-housing-analysis && git pull
    
    # Change to repo directory
    os.chdir(repo_path)
    
    # Add to Python path for module imports
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    
    # Create directories
    (repo_path / 'data/raw').mkdir(parents=True, exist_ok=True)
    (repo_path / 'data/processed').mkdir(parents=True, exist_ok=True)
    (repo_path / 'data/outputs').mkdir(parents=True, exist_ok=True)
    (repo_path / 'databases').mkdir(exist_ok=True)
    (repo_path / 'migrations').mkdir(exist_ok=True)
    
    # Set ROOT for compatibility
    ROOT = repo_path
    
    print(f'\nWorking directory: {os.getcwd()}')
    print(f'Python path updated for module imports')
    
    # Verify modules
    modules_path = repo_path / 'modules'
    if modules_path.exists():
        py_files = [f.name for f in modules_path.glob('*.py') if f.name != '__pycache__']
        print(f'Found {len(py_files)} module files')
    
    print('\nNOTE: You will need to upload your BuildingEye CSV to data/raw/')
    print('Use the file browser on the left, or run:')
    print('  from google.colab import files')
    print('  uploaded = files.upload()')

else:
    # Local environment
    print(f'\nWorking directory: {os.getcwd()}')

print('\n' + '='*70)
print('SETUP COMPLETE!')
print('='*70)

## 1. Setup and Configuration

In [ ]:
import sys
import sqlite3
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# Find project root and setup environment
def find_project_root():
    """Find project root by looking for marker directories."""
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / '00_config').exists() and (path / 'modules').exists():
            return path
    raise FileNotFoundError("Could not find project root")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

# Load config
import json
with open(ROOT / '00_config/berkeley_config.json') as f:
    CONFIG = json.load(f)

# Resolve paths
DATA_DIR = ROOT / CONFIG['paths']['data_dir']
RAW_DIR = ROOT / CONFIG['paths']['raw_dir']
DB_PATH = ROOT / CONFIG['paths']['database']
HOUSING_CSV = ROOT / CONFIG['paths']['housing_projects']

print(f"Project root: {ROOT}")
print(f"Database: {DB_PATH}")
print(f"Raw data: {RAW_DIR}")

In [ ]:
# Import project modules
from modules.data_loader import load_csv, save_to_database
from modules.address_normalizer import normalize_address, standardize_address

## 2. Run Database Migration (if needed)

This adds the new date columns and creates the permit_events table.

In [ ]:
def run_migration(db_path: Path, migration_file: Path):
    """
    Run SQL migration file against the database.
    Handles ALTER TABLE errors gracefully (column already exists).
    """
    print(f"Running migration: {migration_file.name}")
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Read migration SQL
    with open(migration_file) as f:
        sql = f.read()
    
    # Split into statements and execute each
    statements = [s.strip() for s in sql.split(';') if s.strip() and not s.strip().startswith('--')]
    
    success_count = 0
    skip_count = 0
    
    for stmt in statements:
        # Skip comment-only statements
        if all(line.strip().startswith('--') or not line.strip() for line in stmt.split('\n')):
            continue
            
        try:
            cursor.execute(stmt)
            success_count += 1
        except sqlite3.OperationalError as e:
            error_msg = str(e).lower()
            if 'duplicate column' in error_msg or 'already exists' in error_msg:
                skip_count += 1
            else:
                print(f"  Warning: {e}")
                print(f"  Statement: {stmt[:100]}...")
    
    conn.commit()
    conn.close()
    
    print(f"Migration complete: {success_count} statements executed, {skip_count} skipped (already exist)")

# Run the migration
migration_path = ROOT / 'migrations/001_add_timeline_fields.sql'
if migration_path.exists():
    run_migration(DB_PATH, migration_path)
else:
    print(f"Migration file not found: {migration_path}")

## 3. Load BuildingEye CSV Export

Specify the path to your downloaded CSV file.

**In Colab:** Upload your CSV using the file browser or:
```python
from google.colab import files
uploaded = files.upload()  # Select your buildingeye CSV
# Then move it to the right location
!mv *.csv data/raw/
```

In [ ]:
# List available BuildingEye exports
buildingeye_files = list(RAW_DIR.glob('buildingeye*.csv')) + list(RAW_DIR.glob('BuildingEye*.csv'))

if buildingeye_files:
    print("Available BuildingEye exports:")
    for f in sorted(buildingeye_files, key=lambda x: x.stat().st_mtime, reverse=True):
        print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")
    
    # Use most recent by default
    BUILDINGEYE_CSV = sorted(buildingeye_files, key=lambda x: x.stat().st_mtime, reverse=True)[0]
    print(f"\nUsing: {BUILDINGEYE_CSV.name}")
else:
    print("No BuildingEye CSV files found in data/raw/")
    print("Please download from https://berkeley.buildingeye.com/planning")
    print("Save as: data/raw/buildingeye_export_YYYYMMDD.csv")
    BUILDINGEYE_CSV = None

In [ ]:
# Load the CSV
if BUILDINGEYE_CSV and BUILDINGEYE_CSV.exists():
    df_buildingeye = pd.read_csv(BUILDINGEYE_CSV)
    print(f"Loaded {len(df_buildingeye)} records")
    print(f"\nColumns: {df_buildingeye.columns.tolist()}")
    display(df_buildingeye.head(3))
else:
    df_buildingeye = None
    print("No data loaded - please provide a BuildingEye CSV export")

## 4. Parse and Normalize Data

BuildingEye CSV typically has these columns:
- `Address` or `Street Address`
- `Permit Number` or `Agency Reference`
- `Type` or `Permit Type`
- `Received Date` or `Record Date`
- `Status`
- `Description`
- `Assigned Planner`

In [ ]:
def normalize_buildingeye_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize BuildingEye CSV columns to standard names.
    """
    # Common column name mappings
    column_mappings = {
        # Address variations
        'street address': 'address',
        'address': 'address',
        'location': 'address',
        
        # Permit number variations
        'permit number': 'permit_number',
        'permit no': 'permit_number',
        'agency reference': 'permit_number',
        'record number': 'permit_number',
        
        # Type variations
        'type': 'permit_type',
        'permit type': 'permit_type',
        'record type': 'permit_type',
        
        # Date variations
        'received date': 'event_date',
        'record date': 'event_date',
        'date': 'event_date',
        'filed date': 'event_date',
        
        # Status variations
        'status': 'status',
        'record status': 'status',
        
        # Description
        'description': 'description',
        'project description': 'description',
        
        # Assigned
        'assigned planner': 'assigned_to',
        'planner': 'assigned_to',
        'assigned to': 'assigned_to',
    }
    
    # Normalize column names (lowercase, strip whitespace)
    df = df.copy()
    df.columns = [c.lower().strip() for c in df.columns]
    
    # Apply mappings
    df = df.rename(columns=column_mappings)
    
    return df

if df_buildingeye is not None:
    df_normalized = normalize_buildingeye_columns(df_buildingeye)
    print(f"Normalized columns: {df_normalized.columns.tolist()}")

In [ ]:
def parse_buildingeye_dates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Parse date columns from various formats to ISO-8601 (YYYY-MM-DD).
    """
    df = df.copy()
    
    date_columns = [c for c in df.columns if 'date' in c.lower()]
    
    for col in date_columns:
        if col in df.columns:
            # Try multiple date formats
            df[col] = pd.to_datetime(df[col], errors='coerce', infer_datetime_format=True)
            # Convert to ISO format string for SQLite
            df[col] = df[col].dt.strftime('%Y-%m-%d')
            
    return df

if df_buildingeye is not None:
    df_normalized = parse_buildingeye_dates(df_normalized)
    print("Date columns parsed")
    if 'event_date' in df_normalized.columns:
        print(f"Date range: {df_normalized['event_date'].min()} to {df_normalized['event_date'].max()}")

In [ ]:
def identify_permit_type(permit_number: str) -> str:
    """
    Identify permit type from permit number prefix.
    """
    if pd.isna(permit_number):
        return 'Unknown'
    
    permit_upper = str(permit_number).upper().strip()
    
    if permit_upper.startswith('ZP'):
        return 'Zoning'
    elif permit_upper.startswith('BP') or permit_upper.startswith('B20'):
        return 'Building'
    elif permit_upper.startswith('PLN'):
        return 'Planning'
    elif permit_upper.startswith('DEM') or permit_upper.startswith('DP'):
        return 'Demolition'
    elif permit_upper.startswith('CO'):
        return 'CO'
    elif permit_upper.startswith('BL'):
        return 'Business'
    else:
        return 'Other'

def identify_event_type(status: str) -> str:
    """
    Identify event type from status text.
    """
    if pd.isna(status):
        return 'filed'
    
    status_lower = str(status).lower()
    
    if 'approved' in status_lower:
        return 'approved'
    elif 'issued' in status_lower:
        return 'issued'
    elif 'denied' in status_lower or 'rejected' in status_lower:
        return 'denied'
    elif 'expired' in status_lower:
        return 'expired'
    elif 'appeal' in status_lower:
        return 'appealed'
    elif 'final' in status_lower or 'complete' in status_lower or 'occupancy' in status_lower:
        return 'finaled'
    elif 'withdraw' in status_lower:
        return 'withdrawn'
    else:
        return 'filed'  # Default assumption

if df_buildingeye is not None:
    # Add derived columns
    if 'permit_number' in df_normalized.columns:
        df_normalized['permit_type_derived'] = df_normalized['permit_number'].apply(identify_permit_type)
    
    if 'status' in df_normalized.columns:
        df_normalized['event_type'] = df_normalized['status'].apply(identify_event_type)
    else:
        df_normalized['event_type'] = 'filed'
    
    print("Permit types identified:")
    if 'permit_type_derived' in df_normalized.columns:
        print(df_normalized['permit_type_derived'].value_counts())

## 5. Load Existing Projects for Matching

In [ ]:
# Load existing housing projects
df_projects = load_csv(HOUSING_CSV)

if df_projects is not None:
    print(f"Loaded {len(df_projects)} existing projects")
    
    # Create normalized address for matching
    if 'address_display' in df_projects.columns:
        df_projects['address_norm'] = df_projects['address_display'].apply(
            lambda x: normalize_address(str(x)) if pd.notna(x) else ''
        )
    
    # Create permit lookup
    permit_to_project = {}
    for idx, row in df_projects.iterrows():
        if pd.notna(row.get('permits')):
            for permit in str(row['permits']).split(','):
                permit = permit.strip().upper()
                if permit:
                    permit_to_project[permit] = idx
    
    print(f"Created permit lookup with {len(permit_to_project)} permits")

## 6. Match BuildingEye Records to Projects

In [ ]:
def match_to_project(row, df_projects, permit_to_project):
    """
    Match a BuildingEye record to an existing project.
    Returns project_id or None.
    """
    # Try permit number match first
    permit = str(row.get('permit_number', '')).strip().upper()
    if permit in permit_to_project:
        return permit_to_project[permit]
    
    # Try address match
    address = str(row.get('address', '')).strip()
    if address:
        address_norm = normalize_address(address)
        matches = df_projects[df_projects['address_norm'] == address_norm]
        if len(matches) == 1:
            return matches.index[0]
    
    return None

if df_buildingeye is not None and df_projects is not None:
    # Match records
    df_normalized['project_id'] = df_normalized.apply(
        lambda row: match_to_project(row, df_projects, permit_to_project),
        axis=1
    )
    
    matched = df_normalized['project_id'].notna().sum()
    print(f"Matched {matched} of {len(df_normalized)} records ({100*matched/len(df_normalized):.1f}%)")

## 7. Insert into permit_events Table

In [ ]:
def insert_permit_events(df: pd.DataFrame, db_path: Path) -> int:
    """
    Insert permit events into database.
    Returns count of inserted records.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    insert_sql = """
        INSERT OR IGNORE INTO permit_events 
        (project_id, address, permit_number, permit_type, event_type, 
         event_date, status, description, assigned_to, source)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """
    
    inserted = 0
    for _, row in df.iterrows():
        try:
            cursor.execute(insert_sql, (
                row.get('project_id'),
                row.get('address', ''),
                row.get('permit_number', ''),
                row.get('permit_type_derived', row.get('permit_type', 'Unknown')),
                row.get('event_type', 'filed'),
                row.get('event_date'),
                row.get('status', ''),
                row.get('description', '')[:500] if row.get('description') else '',
                row.get('assigned_to', ''),
                'buildingeye'
            ))
            if cursor.rowcount > 0:
                inserted += 1
        except Exception as e:
            print(f"Error inserting {row.get('permit_number')}: {e}")
    
    conn.commit()
    conn.close()
    
    return inserted

if df_buildingeye is not None:
    inserted = insert_permit_events(df_normalized, DB_PATH)
    print(f"Inserted {inserted} permit events into database")

## 8. Update Project Milestone Dates

Calculate and update the milestone dates on housing_projects table.

In [ ]:
def update_project_milestones(db_path: Path):
    """
    Update housing_projects with milestone dates from permit_events.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Update first_filed_date (earliest event date)
    cursor.execute("""
        UPDATE housing_projects
        SET first_filed_date = (
            SELECT MIN(event_date) 
            FROM permit_events 
            WHERE permit_events.project_id = housing_projects.id
        ),
        date_source = 'buildingeye'
        WHERE id IN (SELECT DISTINCT project_id FROM permit_events WHERE project_id IS NOT NULL)
    """)
    print(f"Updated first_filed_date for {cursor.rowcount} projects")
    
    # Update zoning_approved_date
    cursor.execute("""
        UPDATE housing_projects
        SET zoning_approved_date = (
            SELECT MIN(event_date) 
            FROM permit_events 
            WHERE permit_events.project_id = housing_projects.id
              AND permit_events.permit_type = 'Zoning'
              AND permit_events.event_type = 'approved'
        )
        WHERE id IN (SELECT DISTINCT project_id FROM permit_events WHERE project_id IS NOT NULL)
    """)
    print(f"Updated zoning_approved_date for {cursor.rowcount} projects")
    
    # Update building_permit_date
    cursor.execute("""
        UPDATE housing_projects
        SET building_permit_date = (
            SELECT MIN(event_date) 
            FROM permit_events 
            WHERE permit_events.project_id = housing_projects.id
              AND permit_events.permit_type = 'Building'
              AND permit_events.event_type IN ('issued', 'approved')
        )
        WHERE id IN (SELECT DISTINCT project_id FROM permit_events WHERE project_id IS NOT NULL)
    """)
    print(f"Updated building_permit_date for {cursor.rowcount} projects")
    
    # Update co_issued_date and is_completed
    cursor.execute("""
        UPDATE housing_projects
        SET co_issued_date = (
            SELECT MIN(event_date) 
            FROM permit_events 
            WHERE permit_events.project_id = housing_projects.id
              AND (permit_events.permit_type = 'CO' 
                   OR permit_events.event_type = 'finaled'
                   OR permit_events.status LIKE '%Certificate of Occupancy%')
        ),
        is_completed = 1
        WHERE id IN (
            SELECT DISTINCT project_id FROM permit_events 
            WHERE project_id IS NOT NULL
              AND (permit_type = 'CO' 
                   OR event_type = 'finaled'
                   OR status LIKE '%Certificate of Occupancy%')
        )
    """)
    print(f"Updated co_issued_date for {cursor.rowcount} projects")
    
    # Update last_status_date
    cursor.execute("""
        UPDATE housing_projects
        SET last_status_date = (
            SELECT MAX(event_date) 
            FROM permit_events 
            WHERE permit_events.project_id = housing_projects.id
        )
        WHERE id IN (SELECT DISTINCT project_id FROM permit_events WHERE project_id IS NOT NULL)
    """)
    print(f"Updated last_status_date for {cursor.rowcount} projects")
    
    # Calculate total_days for completed projects
    cursor.execute("""
        UPDATE housing_projects
        SET total_days = CAST(
            JULIANDAY(co_issued_date) - JULIANDAY(first_filed_date) AS INTEGER
        )
        WHERE co_issued_date IS NOT NULL AND first_filed_date IS NOT NULL
    """)
    print(f"Calculated total_days for {cursor.rowcount} completed projects")
    
    conn.commit()
    conn.close()

# Run the milestone updates
update_project_milestones(DB_PATH)

## 9. Verify Results

In [ ]:
# Check permit_events table
conn = sqlite3.connect(DB_PATH)

df_events = pd.read_sql_query("""
    SELECT permit_type, event_type, COUNT(*) as count
    FROM permit_events
    GROUP BY permit_type, event_type
    ORDER BY count DESC
""", conn)

print("Permit Events Summary:")
display(df_events)

conn.close()

In [ ]:
# Check project milestone dates
conn = sqlite3.connect(DB_PATH)

df_milestones = pd.read_sql_query("""
    SELECT 
        address_display,
        net_units,
        first_filed_date,
        zoning_approved_date,
        building_permit_date,
        co_issued_date,
        is_completed,
        total_days
    FROM housing_projects
    WHERE first_filed_date IS NOT NULL
    ORDER BY net_units DESC
    LIMIT 20
""", conn)

print("Projects with Timeline Data:")
display(df_milestones)

conn.close()

In [ ]:
# Summary statistics
conn = sqlite3.connect(DB_PATH)

stats = pd.read_sql_query("""
    SELECT 
        COUNT(*) as total_projects,
        SUM(CASE WHEN first_filed_date IS NOT NULL THEN 1 ELSE 0 END) as has_filed_date,
        SUM(CASE WHEN zoning_approved_date IS NOT NULL THEN 1 ELSE 0 END) as has_zoning_date,
        SUM(CASE WHEN building_permit_date IS NOT NULL THEN 1 ELSE 0 END) as has_building_date,
        SUM(CASE WHEN co_issued_date IS NOT NULL THEN 1 ELSE 0 END) as has_co_date,
        SUM(is_completed) as completed_projects,
        AVG(total_days) as avg_days_to_completion
    FROM housing_projects
""", conn)

print("\nTimeline Data Coverage:")
for col in stats.columns:
    print(f"  {col}: {stats[col].iloc[0]}")

conn.close()

## 10. Export Updated Data

Update the housing_projects CSV and sync to Datasette database.

In [ ]:
# Export updated housing_projects to CSV
conn = sqlite3.connect(DB_PATH)
df_updated = pd.read_sql_query("SELECT * FROM housing_projects", conn)
conn.close()

output_csv = DATA_DIR / 'housing_projects_with_dates.csv'
df_updated.to_csv(output_csv, index=False)
print(f"Exported {len(df_updated)} projects to {output_csv}")

---

## Summary

This notebook:
1. Ran database migration to add timeline fields
2. Loaded BuildingEye CSV export
3. Parsed and normalized date fields
4. Matched permits to existing projects
5. Inserted events into permit_events table
6. Updated project milestone dates

**Next Steps:**
- Run `B1_lifecycle_tracking.ipynb` to analyze timelines
- Deploy updated database to Datasette
- View timeline data at https://berkeley-housing.fly.dev/

**To update data:**
1. Download new CSV from https://berkeley.buildingeye.com/planning
2. Save to `data/raw/buildingeye_export_YYYYMMDD.csv`
3. Re-run this notebook